# R2-U2 software regression verification ? Account B T4
Upload only the current `experiments/r2_u2/` folder (including `software_sources/`) to your existing Drive project. No datasets, model weights, full project, or historical evidence are needed.

This notebook stages Git-tracked shared-source files pinned to Git commit c7195dbfa4ccb506bf058102095df92315948d20 plus the experiment code in a unique Colab scratch workspace. Git blob identity and exact transferred byte hashes are checked. The experiment itself is uncommitted and identified by its source hashes. Existing Drive shared sources are never overwritten; an older Trainer is permitted there because tests use the verified scratch snapshot.

Only generated software fixtures run, including one CUDA overflow test. Tiny fixture optimizer updates are intentional; no research fitting, checkpoint acceptance, or promotion is performed. Package installation is never automatic. Run all cells in order in an already prepared Tesla T4 runtime. Full success requires exactly 17 tests with zero skips/errors/failures.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, subprocess, uuid, zipfile, traceback

PROJECT_ROOT = '/content/drive/MyDrive/Colab_R2_U1_Workspace'  # Change only to your actual project location.
ROOT = Path(PROJECT_ROOT).resolve()
EXP = ROOT / 'experiments/r2_u2'
assert EXP.is_dir(), f'Upload experiments/r2_u2 into {ROOT}'
TOKEN = uuid.uuid4().hex
SCRATCH = Path('/content') / ('echoproof-r2-u2-software-' + TOKEN)
assert not SCRATCH.exists()
SCRATCH.mkdir()
(SCRATCH / 'tmp').mkdir()
OUT = EXP / 'reports' / ('software_' + TOKEN)
OUT.mkdir(parents=True, exist_ok=False)
REPORT = {'task': 'R2-U2 software verification', 'started_utc': datetime.now(timezone.utc).isoformat(),
          'research_accepted': False, 'software_gate_passed': False, 'status': 'started',
          'drive_project': str(ROOT), 'scratch': str(SCRATCH)}
def save_report():
    (OUT / 'verification.json').write_text(json.dumps(REPORT, indent=2, sort_keys=True, allow_nan=False))
def sha_bytes(b):
    return hashlib.sha256(b).hexdigest()
CHECKED = False
print('New verification reports:', OUT)
save_report()


Mounted at /content/drive
New verification reports: /content/drive/MyDrive/Colab_R2_U1_Workspace/experiments/r2_u2/reports/software_70fc2d647510481d93297f01ba76665c


## Verify and stage sources
The manifest is the expected identity. Missing or mismatched uploaded files stop execution and produce a failure report. Drive's shared-source differences are recorded, while the Git-pinned snapshot supplies test dependencies. No hash check is waived or rewritten.


In [ ]:
CHECKED = False
try:
    manifest_path = EXP / 'colab_source_manifest.json'
    manifest = json.loads(manifest_path.read_text())
    REPORT['manifest_sha256'] = sha_bytes(manifest_path.read_bytes())
    REPORT['expected_source_hashes'] = manifest['files']
    for name, expected in manifest['files'].items():
        rel = Path(name)
        assert not rel.is_absolute() and '..' not in rel.parts, name
        if name.startswith('experiments/r2_u2/'):
            source = ROOT / rel
            assert source.is_file(), f'Missing upload: {name}'
            assert source.resolve().is_relative_to(EXP), f'Unsafe source path: {name}'
            data = source.read_bytes()
            assert sha_bytes(data) == expected, f'Mismatched upload: {name}'
            if not name.startswith('experiments/r2_u2/software_sources/'):
                target = SCRATCH / rel
                target.parent.mkdir(parents=True, exist_ok=True)
                with target.open('xb') as f: f.write(data)
        else:
            assert name.startswith('src/'), f'Unreviewed manifest dependency: {name}'
    snapshot = json.loads((EXP / 'software_sources/snapshot.json').read_text())
    assert snapshot['git_commit'] == 'c7195dbfa4ccb506bf058102095df92315948d20', 'Unreviewed source snapshot'
    REPORT['shared_source_snapshot'] = snapshot
    with zipfile.ZipFile(EXP / 'software_sources/shared_sources.zip') as bundle:
        assert len(bundle.namelist()) == len(set(bundle.namelist()))
        assert set(bundle.namelist()) == set(snapshot['files'])
        for name, info in snapshot['files'].items():
            rel = Path(name)
            assert not rel.is_absolute() and '..' not in rel.parts
            data = bundle.read(name)
            assert len(data) == info['size'] and sha_bytes(data) == info['sha256'], name
            blob = data if sha_bytes(data) == info['git_content_sha256'] else data.replace(b'\r\n', b'\n')
            assert sha_bytes(blob) == info['git_content_sha256'], name
            assert hashlib.sha1(b'blob ' + str(len(blob)).encode() + b'\0' + blob).hexdigest() == info['git_blob_sha1'], name
            target = SCRATCH / rel
            assert target.resolve().is_relative_to(SCRATCH)
            target.parent.mkdir(parents=True, exist_ok=True)
            with target.open('xb') as f: f.write(data)
    for name, expected in manifest['files'].items():
        if name.startswith('src/'):
            assert sha_bytes((SCRATCH / name).read_bytes()) == expected, f'Snapshot/manifest disagreement: {name}'
    REPORT['existing_drive_shared_source_differences'] = [name for name, expected in manifest['files'].items()
        if name.startswith('src/') and (not (ROOT / name).is_file() or sha_bytes((ROOT / name).read_bytes()) != expected)]
    REPORT['status'] = 'sources_verified'
    CHECKED = True
    print('Verified snapshot; existing Drive shared sources untouched.')
    print('Existing Drive source differences:', REPORT['existing_drive_shared_source_differences'])
except BaseException:
    REPORT.update(status='source_verification_failed', traceback=traceback.format_exc())
    raise
finally:
    save_report()


Verified snapshot; existing Drive shared sources untouched.
Existing Drive source differences: ['src/echoproof/artifacts.py', 'src/echoproof/features/pretrained.py', 'src/echoproof/models/ast_detector.py', 'src/echoproof/training/datasets.py', 'src/echoproof/training/ledger.py', 'src/echoproof/training/seeding.py', 'src/echoproof/training/trainer.py']


## Check the prepared environment and run the 17 tests
The test process imports from scratch only. It has no dataset directory. Fixture runs/ledger remain in scratch. Logs, JUnit and verification JSON persist in the new Drive report directory. If imports, GPU checks or tests fail, retain the report and console output; do not alter sources or install packages to bypass checks.


In [3]:
assert CHECKED, 'Complete source verification first'
import xml.etree.ElementTree as ET
ENV = dict(os.environ, PYTHONPATH=str(SCRATCH / 'src'), PYTHONDONTWRITEBYTECODE='1',
           CUBLAS_WORKSPACE_CONFIG=':4096:8', TMPDIR=str(SCRATCH / 'tmp'),
           TMP=str(SCRATCH / 'tmp'), TEMP=str(SCRATCH / 'tmp'),
           TORCH_HOME=str(SCRATCH / 'torch'), HF_HOME=str(SCRATCH / 'hf'),
           XDG_CACHE_HOME=str(SCRATCH / 'cache'))
try:
    preflight = subprocess.run([sys.executable, '-B', '-c',
        "import sys,json,torch,numpy,pyarrow,pytest; from echoproof.training.trainer import Trainer; "
        "assert torch.cuda.is_available(), 'CUDA unavailable'; "
        "assert torch.cuda.get_device_name(0)=='Tesla T4', 'Select T4'; "
        "assert hasattr(torch.optim.Optimizer,'register_step_post_hook'); "
        "print(json.dumps(dict(python=sys.version,torch=torch.__version__,numpy=numpy.__version__,pyarrow=pyarrow.__version__,pytest=pytest.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0))))"],
        cwd=SCRATCH, env=ENV, capture_output=True, text=True)
    (OUT / 'environment.log').write_text(preflight.stdout + preflight.stderr)
    assert preflight.returncode == 0, preflight.stdout + preflight.stderr
    REPORT['environment'] = json.loads(preflight.stdout.strip().splitlines()[-1])
    print(REPORT['environment'])
    command = [sys.executable, '-B', '-m', 'pytest',
        'experiments/r2_u2/tests/test_contract.py', 'experiments/r2_u2/tests/test_numerical.py', 'experiments/r2_u2/tests/test_launch.py',
        '-q', '-ra', '-p', 'no:cacheprovider', '--basetemp', str(SCRATCH / 'pytest_tmp'),
        '--junitxml=' + str(OUT / 'junit.xml')]
    REPORT['command'] = command
    result = subprocess.run(command, cwd=SCRATCH, env=ENV, capture_output=True, text=True)
    (OUT / 'pytest.log').write_text(result.stdout + result.stderr)
    print(result.stdout, result.stderr)
    cases = list(ET.parse(OUT / 'junit.xml').iter('testcase')) if (OUT / 'junit.xml').exists() else []
    counts = {tag: sum(c.find(tag) is not None for c in cases) for tag in ('failure', 'error', 'skipped')}
    REPORT['test_artifact_sha256'] = {name: sha_bytes((OUT / name).read_bytes()) for name in ('pytest.log', 'junit.xml') if (OUT / name).is_file()}
    REPORT.update(returncode=result.returncode, test_cases=len(cases), counts=counts)
    REPORT['software_gate_passed'] = result.returncode == 0 and len(cases) == 17 and not any(counts.values())
    REPORT['status'] = 'software_verified' if REPORT['software_gate_passed'] else 'software_verification_failed'
    # Recheck both uploaded inputs and staged sources after the fixtures.
    unchanged = all(sha_bytes((ROOT / name).read_bytes()) == expected
                    for name, expected in manifest['files'].items() if name.startswith('experiments/'))
    unchanged = unchanged and all(sha_bytes((SCRATCH / name).read_bytes()) == info['sha256']
                                for name, info in snapshot['files'].items())
    REPORT['verification_inputs_unchanged'] = unchanged
    if not unchanged:
        REPORT.update(software_gate_passed=False, status='source_integrity_failed')
    print('Software gate passed:', REPORT['software_gate_passed'])
except BaseException:
    REPORT.update(status='environment_or_execution_failed', software_gate_passed=False, traceback=traceback.format_exc())
    print(REPORT['traceback'])
finally:
    REPORT['finished_utc'] = datetime.now(timezone.utc).isoformat()
    save_report()
    print('Persistent report:', OUT)
print('This is software verification only; no research result is accepted.')


Traceback (most recent call last):
  File "/tmp/ipykernel_1592/1718114579.py", line 17, in <cell line: None>
    assert preflight.returncode == 0, preflight.stdout + preflight.stderr
           ^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import sys,json,torch,numpy,pyarrow,pytest; from echoproof.training.trainer import Trainer; assert torch.cuda.is_available(), 'CUDA unavailable'; assert torch.cuda.get_device_name(0)=='Tesla T4', 'Select T4'; assert hasattr(torch.optim.Optimizer,'register_step_post_hook'); print(json.dumps(dict(python=sys.version,torch=torch.__version__,numpy=numpy.__version__,pyarrow=pyarrow.__version__,pytest=pytest.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0))))
                                                                                                       ~~~~~~~~~~~~~~~~~~~~~~~^^
AssertionError: CUDA unavailable


Persistent report: /content/drive/MyDrive/Co

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU: Tesla T4


## Download results
Run even after a failed check. Send this small ZIP back for review. A successful gate means 17 passed and zero skipped, errors or failures; it does not authorize research training. Do not rerun the historical warm-up notebook.


In [ ]:
from google.colab import files
RESULT_ZIP = SCRATCH / 'software_verification_results.zip'
with zipfile.ZipFile(RESULT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUT.iterdir()):
        if path.is_file(): bundle.write(path, path.name)
files.download(str(RESULT_ZIP))
